# 4. Deploy Parsing Endpoints

This notebook handles deployment of the document parsing serving endpoint. This allows you to process documents via API calls.

**What this deploys:**
- Docling serving endpoint for on-demand document processing
- MLflow model with GPU support (optional)
- REST API for document processing requests

In [ ]:
# Deploy the Docling serving endpoint
import logging
import mlflow
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import EndpointCoreConfigInput, ServedEntityInput

# Configure logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

def register_model():
    """Register the Docling parsing model in MLflow Model Registry."""
    logger.info("Registering Docling parsing model")
    
    # Set MLflow registry to Unity Catalog
    mlflow.set_registry_uri("databricks-uc")
    
    with mlflow.start_run(run_name="docling_endpoint_registration"):
        # Import the serving endpoint model
        from src.docling_endpoint import DoclingParsingModel
        
        # Create model instance
        docling_model = DoclingParsingModel()
        
        model_info = mlflow.pyfunc.log_model(
            artifact_path="docling_parser",
            python_model=docling_model,
            conda_env={
                "channels": ["conda-forge"],
                "dependencies": [
                    "python=3.12.3",
                    "pip",
                    {
                        "pip": [
                            "docling>=2.68.0",
                            "databricks-sdk>=0.28.0",
                            "mlflow>=3.1",
                            "pillow>=10.3.0",
                        ]
                    }
                ]
            },
            pip_requirements=[
                "docling>=2.68.0",
                "databricks-sdk>=0.28.0", 
                "mlflow>=3.1",
                "pillow>=10.3.0",
            ]
        )
        
        logger.info(f"Model logged with URI: {model_info.model_uri}")
        return model_info

def deploy_endpoint():
    """Deploy the serving endpoint."""
    logger.info("Starting endpoint deployment")
    
    # Register model first
    model_info = register_model()
    
    # Register in Unity Catalog
    model_name = "main.default.docling_parser"
    
    registered_model = mlflow.register_model(
        model_uri=model_info.model_uri,
        name=model_name,
        tags={"purpose": "document_parsing", "framework": "docling"}
    )
    
    logger.info(f"Model registered: {model_name}, version: {registered_model.version}")
    
    # Deploy serving endpoint
    w = WorkspaceClient()
    endpoint_name = "docling-document-parser"
    
    try:
        endpoint = w.serving_endpoints.create(
            name=endpoint_name,
            config=EndpointCoreConfigInput(
                served_entities=[
                    ServedEntityInput(
                        name="docling-parser",
                        entity_name=model_name,
                        entity_version=registered_model.version,
                        workload_size="Small",
                        workload_type="GPU_SMALL", 
                        scale_to_zero_enabled=True
                    )
                ]
            )
        )
        
        print(f"✅ Serving endpoint created: {endpoint_name}")
        print(f"🌐 Endpoint URL: {endpoint.endpoint_url}")
        return endpoint_name
        
    except Exception as e:
        print(f"❌ Endpoint deployment failed: {e}")
        print("You can deploy manually using the Databricks UI")
        return None

# Run deployment
endpoint_name = deploy_endpoint()
if endpoint_name:
    print(f"🎉 Docling serving endpoint '{endpoint_name}' deployed successfully!")

## Test the deployed endpoint

Let's test the deployed endpoint to make sure it's working:

In [ ]:
# Test the serving endpoint
from mlflow.deployments import get_deploy_client
from pathlib import Path

# Test with a sample document
raw_doc_dir = "/Volumes/main/default/raw_docs"
test_docs = list(Path(raw_doc_dir).glob("*.pdf"))

if test_docs:
    test_doc = str(test_docs[0])
    
    print(f"🧪 Testing endpoint with: {Path(test_doc).name}")
    
    try:
        # Get deployment client
        deploy_client = get_deploy_client('databricks')
        
        # Test request
        request = {
            "dataframe_split": {
                "columns": ["file_path"],
                "data": [[test_doc]]
            }
        }
        
        # Call endpoint (update with your actual endpoint name)
        endpoint_name = "docling-parsing-endpoint"  # Update this
        response = deploy_client.predict(endpoint=endpoint_name, inputs=request)
        
        print("✅ Endpoint test successful!")
        print(f"📊 Response keys: {list(response.keys())}")
        
        if 'pages' in response:
            print(f"  Pages processed: {response['pages']}")
        if 'pictures' in response:
            print(f"  Pictures found: {response['pictures']}")
        if 'tables' in response:
            print(f"  Tables found: {response['tables']}")
            
    except Exception as e:
        print(f"❌ Endpoint test failed: {e}")
        print("Make sure the endpoint is deployed and the name is correct")

else:
    print("❌ No test documents found. Run 1_setup.ipynb first.")

## Deployment Complete! 

Your document parsing serving endpoint is now ready to use.

### What was deployed:
✅ **Docling MLflow Model** - Packaged for serving
✅ **Serving Endpoint** - REST API for document processing  
✅ **GPU Support** - For faster processing (if configured)
✅ **Auto-scaling** - Scales based on demand

### Usage Examples:

**Via Python Client:**
```python
from mlflow.deployments import get_deploy_client

deploy_client = get_deploy_client('databricks')
response = deploy_client.predict(
    endpoint="docling-parsing-endpoint",
    inputs={"dataframe_split": {"columns": ["file_path"], "data": [["/path/to/doc.pdf"]]}}
)
```

**Via REST API:**
```bash
curl -X POST https://your-workspace.cloud.databricks.com/serving-endpoints/docling-parsing-endpoint/invocations \
  -H "Authorization: Bearer $DATABRICKS_TOKEN" \
  -H "Content-Type: application/json" \
  -d '{"dataframe_split": {"columns": ["file_path"], "data": [["/path/to/doc.pdf"]]}}'
```

### Next Steps:
- Use the endpoint in production applications
- Monitor endpoint performance and scaling
- Process documents on-demand via API calls